# Индексация, broadcasting и векторизация

Авторы BelOAI. CC BY 4.0.

Выполняйте ячейки сверху вниз. Сначала решите задачи, затем обратитесь к решениям.

## Чему научимся

Проверять формы операций, выбирать строки маской и вычислять расстояния векторно.

## Данные для примеров

Строки таблицы — три объекта, столбцы — размах и масса. Создадим массив заново, чтобы примеры можно было выполнить в отдельном ноутбуке.

In [ ]:
import numpy as np
X = np.array([[12., 8.], [22., 10.], [14., 9.]])


## Индексация, маски и копии

В `X[строки, столбцы]` двоеточие выбирает всю ось. Логическая маска хранит `True` для нужных строк. Маски объединяют `&`, `|`, `~` со скобками: «и», «или», «не». Например, `(X[:, 0] >= 15) & (X[:, 1] < 11)`. Операторы `and` и `or` для целых массивов не подходят: массив содержит много логических значений.

Обычный срез массива часто даёт **представление**: оно использует ту же память, что и исходный массив. Изменение элементов представления меняет исходный массив. Выбор логической маской создаёт копию. Для независимости явно используйте `.copy()`. Функция `np.shares_memory(a, b)` проверяет, есть ли у массивов общая память.

In [ ]:
print(X[:, 0])             # [12. 22. 14.]
mask = X[:, 0] >= 15
selected = X[mask]
assert selected.shape == (1, 2)
view = X[:1]
independent = X[:1].copy()
assert np.shares_memory(view, X)
assert not np.shares_memory(independent, X)


## Broadcasting: согласование форм

При поэлементной операции NumPy сравнивает размеры справа налево. Размеры совместимы, если равны или один из них равен 1; отсутствующие слева оси считаются единичными. Такое согласование называют broadcasting. Например, `(3, 2)` и `(2,)` согласуются как `(3, 2)` и `(1, 2)`: один вектор вычитается из каждой строки.

Метод `reshape` меняет форму при сохранении числа элементов. `np.allclose` проверяет приблизительное равенство чисел: абсолютный допуск ограничивает малую разность, относительный учитывает масштаб чисел. В наших примерах достаточно стандартных допусков, но при очень маленьких величинах их выбирают отдельно. Сначала проверяйте формы: `allclose` тоже применяет broadcasting и не доказывает совпадения форм. Для точного сравнения массивов одинаковой формы есть `np.array_equal`.

In [ ]:
center = X.mean(axis=0)
centered = X - center
assert centered.shape == (3, 2)
assert np.allclose(centered.mean(axis=0), [0, 0])
a = np.array([1., 2., 3.])
column = a.reshape(3, 1)
assert (column - a).shape == (3, 3)


Последняя операция — частая ошибка: `(3, 1)` и `(3,)` согласуются как `(3, 1)` и `(1, 3)`. Получаются все девять разностей, а не три разности соответствующих элементов. Перед операцией полезно выписать формы на бумаге.

Обучающая часть, или `train`, — объекты, доступные при настройке модели; проверочная часть — объекты для измерения её качества. При подготовке модели среднее вычисляют только на обучающих строках, затем вычитают **то же** среднее из проверочных. Если сначала вычислить среднее всей таблицы, подготовка использует информацию о проверочной части: это утечка данных. Центрирование выше показывает арифметику; для оценки модели таблицу сначала разделяют.

## Разобранный пример: расстояния без цикла Python

Вычтем координаты запроса из каждой строки, возведём разности в квадрат и сложим по координатам. `np.argmin` даёт индекс первого наименьшего значения. Квадрата расстояния достаточно для сравнения соседей; если нужна сама длина, `np.sqrt` извлечёт квадратный корень каждого элемента. Как в математике, `@` умножает матрицу на вектор, а `*` — элементы. Для строки `[12, 8]` и весов `[2, -1]` матрично-векторное произведение даёт `12 * 2 + 8 * (-1) = 16`: по одному результату на строку.

In [ ]:
query = np.array([15., 9.])
differences = X - query
squared_distances = (differences ** 2).sum(axis=1)
assert np.allclose(squared_distances, [10, 50, 1])
assert int(np.argmin(squared_distances)) == 2
weights = np.array([2., -1.])
assert np.allclose(X @ weights, [16, 34, 19])


Для первой строки получаем $(12-15)^2+(8-9)^2=10$. Это **векторизация**: работа передана библиотечным операциям. Она не отменяет расход памяти — здесь создаётся таблица разностей. Единицы признаков всё ещё влияют на результат, как в [уроке о масштабе признаков](../../../math/linear-algebra/feature-scale/).

## Случайность и точность

`np.random.default_rng(seed)` создаёт генератор случайных чисел с заданным начальным состоянием — seed. Метод `random` выдаёт числа от 0 включительно до 1 не включительно в массиве указанной формы. `np.isclose` сравнивает отдельные числа с допуском.

In [ ]:
rng = np.random.default_rng(17)
noise = rng.random(size=(3, 2))
assert noise.shape == X.shape
assert np.isclose(0.1 + 0.2, 0.3)


Повторный вызов того же генератора выдаст следующие числа; для повторения создайте новый генератор с тем же seed в том же окружении. Дробные вычисления сравнивайте с допуском: точное равенство `0.1 + 0.2 == 0.3` обычно ложно.

Для справки: [руководство NumPy](https://numpy.org/doc/stable/user/quickstart.html), [сравнение с допуском allclose](https://numpy.org/doc/stable/reference/generated/numpy.allclose.html).

## Задачи

### 1. Базовая

Что выбирает `X[:, 0]`?

<details>
<summary>Подсказка</summary>

Первая часть индекса выбирает строки.

</details>



### 2. Базовая

Чем опасно изменение обычного среза массива?

<details>
<summary>Подсказка</summary>

Он может разделять память.

</details>



### 3. Прикладная

Выберите строки с размахом не меньше 14 и массой не больше 9.

<details>
<summary>Подсказка</summary>

Объедините две маски скобками и &.

</details>



### 4. Прикладная

Найдите длины векторов (3, 4) и (5, 12) без цикла Python.

<details>
<summary>Подсказка</summary>

Сумма квадратов по координатам и np.sqrt.

</details>



### 5. Олимпиадная

Почему разность массивов (100,) и (100, 1) не даёт 100 соответствующих ошибок?

<details>
<summary>Подсказка</summary>

Сравните размеры справа налево.

</details>



### 6. Олимпиадная

Почему среднее перед центрированием модели считают только по train?

<details>
<summary>Подсказка</summary>

Статистика проверочных объектов тоже несёт информацию.

</details>



## Итоги

Вы умеете выбирать строки маской, проверять broadcasting и вычислять расстояния операциями над массивами.

## Самопроверка

Выпишите форму результата для `(3, 2) - (2,)` и `(3, 1) - (3,)`. Почему перед `allclose` нужна отдельная проверка формы? Чем срез отличается от выбора маской по общей памяти?

Для перехода дальше достаточно уверенно решать базовые и прикладные задачи. К олимпиадным можно вернуться позже.


## Решения задач

### Задача 1

Первый столбец всех строк.

### Задача 2

Изменение может затронуть исходный массив.

### Задача 3

In [ ]:
chosen = X[(X[:, 0] >= 14) & (X[:, 1] <= 9)]
assert np.array_equal(chosen, [[14, 9]])


### Задача 4

In [ ]:
vectors = np.array([[3., 4.], [5., 12.]])
assert np.allclose(np.sqrt((vectors ** 2).sum(axis=1)), [5, 13])


### Задача 5

Broadcasting создаёт (100, 100), то есть все пары. Нужно согласовать формы до вычитания.

### Задача 6

Иначе подготовка использует проверочную часть и создаёт утечку. Обучающее среднее применяют к остальным частям без пересчёта.